In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,classification_report
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.svm import SVC

In [12]:
url = "https://users.stat.ufl.edu/~winner/data/armada.dat"
df = pd.read_fwf(url)
df

,Bantam,1601,6,3,0,2,0.1,0.2
0,Malacca Strait,1606,14,11,0,1.273,0,0
1,Ilha das Naus,1606,6,9,0,0.667,0,-1
2,Pulo Butum,1606,7,9,0,0.778,0,1
3,Surrat,1615,6,0,4,1.500,0,0
4,Ilha das Naus,1615,3,5,0,0.600,0,-1
5,Jask,1620,4,0,4,1.000,0,0
6,Hormuz,1622,6,0,5,1.200,0,-1
7,Mogincoal Shoals,1622,4,4,2,0.667,0,-1
8,Hormuz,1625,8,4,4,1.000,0,0
9,Goa,1636,6,4,0,1.500,0,0


In [13]:
#Adding column names to the dataset collected from the url:

df.columns = [
    "Battle",
    "Year",
    "Portuguese_ships",
    "Dutch_ships",
    "English_ships",
    "Ratio_of_P", #Ratio of Portuguese ships to Dutch and British ships
    "Spanish_invol", #Spanish involvement (1 = Yes and 0 = No
    "P_outcome" #Portuguese outcome (-1 = Deafeat, 0 = Draw, 1 = Victory)
]

df

,Battle,Year,Portuguese_ships,Dutch_ships,English_ships,Ratio_of_P,Spanish_invol,P_outcome
0,Malacca Strait,1606,14,11,0,1.273,0,0
1,Ilha das Naus,1606,6,9,0,0.667,0,-1
2,Pulo Butum,1606,7,9,0,0.778,0,1
3,Surrat,1615,6,0,4,1.500,0,0
4,Ilha das Naus,1615,3,5,0,0.600,0,-1
5,Jask,1620,4,0,4,1.000,0,0
6,Hormuz,1622,6,0,5,1.200,0,-1
7,Mogincoal Shoals,1622,4,4,2,0.667,0,-1
8,Hormuz,1625,8,4,4,1.000,0,0
9,Goa,1636,6,4,0,1.500,0,0


In [4]:
#Checking the data if there are missing values:

df.isna().sum()

Battle              0
Year                0
Portuguese_ships    0
Dutch_ships         0
English_ships       0
Ratio_of_P          0
Spanish_invol       0
P_outcome           0
dtype: int64

In [5]:
df.shape

(27, 8)

In [6]:
#Mapping the P_outcome to 1,2,3 for SVM:

outcome_map = {-1:0, 0:1, 1:2} #-1 (losses) to 0, 0 (draws) to 1, and 1 (wins) to 2
df["P_outcome_clean"] = df["P_outcome"].map(outcome_map)

In [7]:
df.head()

,Battle,Year,Portuguese_ships,Dutch_ships,English_ships,Ratio_of_P,Spanish_invol,P_outcome,P_outcome_clean
0,Malacca Strait,1606,14,11,0,1.273,0,0,1
1,Ilha das Naus,1606,6,9,0,0.667,0,-1,0
2,Pulo Butum,1606,7,9,0,0.778,0,1,2
3,Surrat,1615,6,0,4,1.500,0,0,1
4,Ilha das Naus,1615,3,5,0,0.600,0,-1,0


In [8]:
df['P_outcome_clean'].value_counts()

P_outcome_clean
1    12
0    10
2     5
Name: count, dtype: int64

1) Use an SVM-based model to predict the Portuguese outcome of the battle from the number of ships involved on all sides and Spanish involvement.

In [9]:
#Defining X and y to conduct SVM model:

X = df[['Portuguese_ships','Dutch_ships','English_ships','Spanish_invol']]
y = df['P_outcome_clean']

#Training the dataset and testing 30% of the data:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3, random_state=1)

#standardizing the data:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Using SVM model:
model = SVC(kernel = 'rbf', C=1, gamma='scale', decision_function_shape = "ovo",  class_weight='balanced') #Did a one v. one approach to create a binary classifer for each class
model.fit(X_train_scaled,y_train)

# Making predictions from the SVC model:
y_pred = model.predict(X_test_scaled)

print(accuracy_score(y_test,y_pred))
print(classification_report(y_test, y_pred))

0.4444444444444444
              precision    recall  f1-score   support

           0       0.75      0.75      0.75         4
           1       1.00      0.20      0.33         5
           2       0.00      0.00      0.00         0

    accuracy                           0.44         9
   macro avg       0.58      0.32      0.36         9
weighted avg       0.89      0.44      0.52         9



/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

multiclass SVM source: https://www.geeksforgeeks.org/machine-learning/multi-class-classification-using-support-vector-machines-svm/

SVM results:
    Accuracy score of predicting the outcome of Portuguese is 44% whether it is a victory, draw or loss. The SVM had difficulty predicting wins and while looking at the number of wins, the Portuguese had 5 recorded wins. The SVM model was able to predict a bit more accurately the number of draws and losses from this small sample size.

2) Try solving the same problem using two other classifiers that you know.

Will be conducting a multinomial logistic regression model:

In [10]:
from sklearn.linear_model import LogisticRegression

X = df[['Portuguese_ships','Dutch_ships','English_ships','Spanish_invol']]
y = df['P_outcome_clean']

#Splitting the data:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3, random_state=1)

#Scaling the data:
scaler = StandardScaler()
X_train_scaled1 = scaler.fit_transform(X_train)
X_test_scaled1 = scaler.transform(X_test)

# Logistic Regression (multinomial)
multi_reg = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=1000,
    class_weight='balanced'  # balancing the class weight since the dataset is small
)

#Predicting the dataset:

multi_reg.fit(X_train_scaled1, y_train)
y_pred = multi_reg.predict(X_test_scaled1)

print(classification_report(y_test, y_pred))
print(accuracy_score(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.60      0.75      0.67         4
           1       1.00      0.20      0.33         5
           2       0.00      0.00      0.00         0

    accuracy                           0.44         9
   macro avg       0.53      0.32      0.33         9
weighted avg       0.82      0.44      0.48         9

0.4444444444444444


/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.

From the multinomial logistic regression the accuracy score of the model is still 44% and it had a difficult time to predict the Portuguese outcome of wins (classified as 2). The wins had precision, recall, and f1 score of 0.0. 

Will conduct a random forest to observe if the model can predict the outcome of Portuguese wins, losses, or draws. 

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

#Defining parameters:
X_rf = df[['Portuguese_ships', 'Dutch_ships', 'English_ships', 
        'Ratio_of_P', 'Spanish_invol']]
y_rf = df['P_outcome_clean']

#Training the data:
X_train, X_test, y_train, y_test = (train_test_split(X_rf,y_rf,test_size=0.3, random_state=1))

#Random Forest model:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced', #dataset is small
    random_state=42
)

#Cross-validation:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred = cross_val_predict(rf, X_rf, y_rf, cv=cv)

#Results:
print(accuracy_score(y, y_pred))
print(classification_report(y, y_pred))

0.37037037037037035
              precision    recall  f1-score   support

           0       0.56      0.50      0.53        10
           1       0.36      0.42      0.38        12
           2       0.00      0.00      0.00         5

    accuracy                           0.37        27
   macro avg       0.30      0.31      0.30        27
weighted avg       0.36      0.37      0.37        27



The accuracy score from the random forest was 37% and is unable to predict number of wins for the Portuguese due to a small dataset.

3) Report and compare their results with those from SVM.

When conducting a multinomial logistic regression model, the accuracy score was 44% which was the same as the SVM model. Since the dataset was small with only 28 entries, the total number of wins recorded for the Portuguese had a total of 5 wins which can be difficult to predict from the dataset.  SVM and the logistic regression models, were testing 30% percent of the data which is approximately 9 samples being tested which can probably explain their accuracy scores being low. However, the classifcation report from the SVM model presented better precison for losses and draws compared to the logistic regression model. The precision value for SVM in losses were 0.75 and in draws were 1.00 while the precision value for losses in the regression model was 0.60 and the draws were 1.00. The SVM model was able to accurately predict losses more accurately since there were 12 recorded losses from the dataset. 
As for the random forest, the accuracy score was 37% making it the lowest compared to the regression and SVM models. Even though the cross validation was added to the model, it still had a difficult time to predict the outcome of Portuguese losses, wins, or draws from the dataset. If the dataset would have been larger it will be able to predict the outcome of wins, losses, and draws a bit more accurately.